In [1]:
import tensorflow as tf
        
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

I0000 00:00:1786720959.234412     575 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786720960.248026     575 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786720961.860298     575 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

In [3]:
# ==========================================
# 1. GPU VRAM Safety Protocol
# ==========================================
# Forces TensorFlow to only use the VRAM it needs, preventing OOM crashes
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU Memory Growth Enabled")
    except RuntimeError as e:
        print(e)

GPU Memory Growth Enabled


In [10]:
# ==========================================
# 1. CONFIGURATION & HYPERPARAMETERS
# ==========================================
CSV_FILE = r"../Dataset/CleanedDataSetNew.csv"                  # Your dataset CSV
IMAGE_DIR = r"../Dataset/AllImages"          # Folder containing the ESP32-CAM images
TARGET_COLUMN = "irradiance"           # The CSV column with your W/m^2 values
IMAGE_FILENAME_COLUMN = "dateTime"     # The CSV column with your image names

IMG_SIZE = 380                         # Required for EfficientNetB4
BATCH_SIZE = 6                        # Keep low (8 or 16) if running on laptop CPU/GPU
EPOCHS = 30                            # Maximum training iterations
LEARNING_RATE = 0.001


In [11]:
# ==========================================
# 2. DATASET PREPARATION (Pandas) - UPDATED
# ==========================================
def prepare_dataframe():
    print("📊 Loading CSV dataset...")
    df = pd.read_csv(CSV_FILE)
    
    # 🚨 FIX 1: Force Pandas to treat the filename/dateTime column as strings
    df[IMAGE_FILENAME_COLUMN] = df[IMAGE_FILENAME_COLUMN].astype(str)
    
    # Get list of physical image files
    image_list = [filename[:12] for filename in os.listdir(IMAGE_DIR)]
    
    # Filter dataset to only include rows where the image actually exists
    df["ImageBool"] = df[IMAGE_FILENAME_COLUMN].isin(image_list)
    df = df[df["ImageBool"] == True]
    
    # Construct the full file paths
    df["file_path"] = df[IMAGE_FILENAME_COLUMN].apply(
        lambda x: os.path.join(IMAGE_DIR, f"{x}.jpg") # Adjust to .png if needed
    )
    
    # 🚨 FIX 2: Explicitly cast arrays to string and float32
    file_paths = df["file_path"].astype(str).values
    targets = df[TARGET_COLUMN].astype("float32").values
    
    print(f"✅ Found {len(df)} valid image-irradiance pairs.")
    return file_paths, targets

In [12]:



# ==========================================
# 3. TENSORFLOW DATA PIPELINE - UPDATED
# ==========================================
def process_path(file_path, irradiance):
    # Read the physical file
    img = tf.io.read_file(file_path)
    # Decode JPEG to a tensor
    img = tf.image.decode_jpeg(img, channels=3)
    # Resize to EfficientNetB4 specs (380x380)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    
    return img, irradiance

def create_tf_dataset(file_paths, targets):
    # 🚨 FIX 3: Cast dataset inputs to tf.string and tf.float32 explicitly
    dataset = tf.data.Dataset.from_tensor_slices((
        tf.cast(file_paths, tf.string),
        tf.cast(targets, tf.float32)
    ))
    
    dataset = dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.shuffle(buffer_size=1000)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

In [13]:
# ==========================================
# 4. MODEL ARCHITECTURE
# ==========================================
def build_irradiance_model():
    print("⚙️ Building EfficientNetB4 Regression Architecture...")
    
    base_model = EfficientNetB4(
        weights='imagenet', 
        include_top=False, 
        input_shape=(IMG_SIZE, IMG_SIZE, 3) 
    )
    
    # Freeze the base model to train only the custom regression head first
    base_model.trainable = False 

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3), 
        
        # Single output neuron with ReLU (Irradiance cannot be negative)
        layers.Dense(1, activation='relu', name='irradiance_output') 
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.Huber(),
        metrics=['mae'] 
    )
    
    return model

In [14]:
paths, targets = prepare_dataframe()

📊 Loading CSV dataset...
✅ Found 44048 valid image-irradiance pairs.


In [19]:
BASE_DIR = os.path.dirname(os.getcwd())

INPUT_PATH = os.path.join(BASE_DIR, "Dataset/Download Images")
OUTPUT_PATH = os.path.join(BASE_DIR, "Dataset/AllImages")
CSV_FILE = os.path.join(BASE_DIR, "Dataset/DataSet.csv")
def check():
    files = os.listdir(OUTPUT_PATH)
    #print(files)
    Type = set()  
    z = []
    for i in files:
        Type.add(i[12:])
        if i[12:] == ".zip":
            z.append(i)

    print(Type)

check()

{'.jpg'}


In [15]:
# ==========================================
# 5. EXECUTION & TRAINING LOOP
# ==========================================
if __name__ == "__main__":
    # 1. Prepare data
    paths, targets = prepare_dataframe()
    
    # Optional: Split into training and validation sets (80/20 split)
    split_index = int(len(paths) * 0.8)
    train_paths, val_paths = paths[:split_index], paths[split_index:]
    train_targets, val_targets = targets[:split_index], targets[split_index:]
    
    # 2. Build TensorFlow Datasets
    print("🚀 Building TensorFlow Data Pipelines...")
    train_dataset = create_tf_dataset(train_paths, train_targets)
    val_dataset = create_tf_dataset(val_paths, val_targets)
    
    # 3. Build Model
    model = build_irradiance_model()
    model.summary()
    
    # 4. Set up Callbacks
    # Saves the best version of the model automatically
    checkpoint = ModelCheckpoint(
        "efficientnet_b4_solar.keras", 
        monitor="val_loss", 
        save_best_only=True, 
        verbose=1
    )
    # Stops training early if the model stops improving, saving time
    early_stop = EarlyStopping(
        monitor="val_loss", 
        patience=5, 
        restore_best_weights=True
    )
    
    # 5. Train the Model
    print("🔥 Starting Training...")
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=EPOCHS,
        callbacks=[checkpoint, early_stop]
    )
    
    print("🎉 Training Complete! Model saved as 'efficientnet_b4_solar.keras'")

📊 Loading CSV dataset...
✅ Found 44048 valid image-irradiance pairs.
🚀 Building TensorFlow Data Pipelines...
⚙️ Building EfficientNetB4 Regression Architecture...


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb4 (Functional)     │ (None, 12, 12, 1792)   │    17,673,823 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1792)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       459,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ irradiance_output (Dense)       │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,133,088 (69.17 MB)

 Trainable params: 459,265 (1.75 MB)

 Non-trainable params: 17,673,823 (67.42 MB)

🔥 Starting Training...
Epoch 1/30


/home/chinmay/tf-gpu/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1786721063.075058     689 shuffle_dataset_op.cc:453] ShuffleDatasetV3:12: Filling up shuffle buffer (this may take a while): 672 of 1000
I0000 00:00:1786721068.065168     689 shuffle_dataset_op.cc:483] Shuffle buffer filled.
I0000 00:00:1786721068.730529     631 service.cc:153] XLA service 0x7eb59c0b14f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1786721068.731221     631 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6 (Driver: 12.7.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1786721073.550851     631 dump_mlir_util.cc:26

5873/5873 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 98.2632 - mae: 98.7585
Epoch 1: val_loss improved from None to 97.59549, saving model to efficientnet_b4_solar.keras

Epoch 1: finished saving model to efficientnet_b4_solar.keras
5873/5873 ━━━━━━━━━━━━━━━━━━━━ 613s 91ms/step - loss: 98.2632 - mae: 98.7585 - val_loss: 97.5955 - val_mae: 98.0938
Epoch 2/30
5873/5873 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 74.3486 - mae: 74.8420
Epoch 2: val_loss improved from 97.59549 to 75.92367, saving model to efficientnet_b4_solar.keras

Epoch 2: finished saving model to efficientnet_b4_solar.keras
5873/5873 ━━━━━━━━━━━━━━━━━━━━ 484s 81ms/step - loss: 74.3486 - mae: 74.8420 - val_loss: 75.9237 - val_mae: 76.4220
Epoch 3/30
5873/5873 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 69.7780 - mae: 70.2711
Epoch 3: val_loss improved from 75.92367 to 68.07067, saving model to efficientnet_b4_solar.keras

Epoch 3: finished saving model to efficientnet_b4_solar.keras
5873/5873 ━━━━━━━━━━━━━━━━━━━━ 471s 80ms/s

In [20]:
bad_images = []

for i, path in enumerate(paths):
    try:
        img = tf.io.read_file(path)
        tf.io.decode_jpeg(img, channels=3)
    except Exception as e:
        bad_images.append((path, str(e)))

print("BAD IMAGES:", len(bad_images))

for x in bad_images[:20]:
    print(x)

I0000 00:00:1786687892.902142    2021 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1768 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6
E0000 00:00:1786687978.031271    2021 jpeg_mem.cc:331] Premature end of JPEG data. Stopped at line 1518/1536
W0000 00:00:1786687978.031785    2021 local_rendezvous.cc:412] Local rendezvous is aborting with status: INVALID_ARGUMENT: jpeg::Uncompress failed. Invalid JPEG data or crop window.
E0000 00:00:1786687988.468619    2021 jpeg_mem.cc:331] Premature end of JPEG data. Stopped at line 1422/1536
W0000 00:00:1786687988.468663    2021 local_rendezvous.cc:412] Local rendezvous is aborting with status: INVALID_ARGUMENT: jpeg::Uncompress failed. Invalid JPEG data or crop window.
E0000 00:00:1786688000.692755    2021 jpeg_mem.cc:331] Premature end of JPEG data. Stopped at line 1150/1536
E0000 00:00:1786688001.427280    2021 jpeg_mem.cc:331] Prematur

BAD IMAGES: 19
('../Dataset/AllImages/202405230540.jpg', '{{function_node __wrapped__DecodeJpeg_device_/job:localhost/replica:0/task:0/device:CPU:0}} jpeg::Uncompress failed. Invalid JPEG data or crop window. [Op:DecodeJpeg]')
('../Dataset/AllImages/202406080600.jpg', '{{function_node __wrapped__DecodeJpeg_device_/job:localhost/replica:0/task:0/device:CPU:0}} jpeg::Uncompress failed. Invalid JPEG data or crop window. [Op:DecodeJpeg]')
('../Dataset/AllImages/202406271200.jpg', '{{function_node __wrapped__DecodeJpeg_device_/job:localhost/replica:0/task:0/device:CPU:0}} jpeg::Uncompress failed. Invalid JPEG data or crop window. [Op:DecodeJpeg]')
('../Dataset/AllImages/202406281240.jpg', '{{function_node __wrapped__DecodeJpeg_device_/job:localhost/replica:0/task:0/device:CPU:0}} jpeg::Uncompress failed. Invalid JPEG data or crop window. [Op:DecodeJpeg]')
('../Dataset/AllImages/202406301010.jpg', '{{function_node __wrapped__DecodeJpeg_device_/job:localhost/replica:0/task:0/device:CPU:0}} jp

In [22]:
for path, error in bad_images:
    if os.path.exists(path):
        os.remove(path)
        print("Deleted:", path)

print(f"\nDeleted {len(bad_images)} corrupted images.")

Deleted: ../Dataset/AllImages/202405230540.jpg
Deleted: ../Dataset/AllImages/202406080600.jpg
Deleted: ../Dataset/AllImages/202406271200.jpg
Deleted: ../Dataset/AllImages/202406281240.jpg
Deleted: ../Dataset/AllImages/202406301010.jpg
Deleted: ../Dataset/AllImages/202407011340.jpg
Deleted: ../Dataset/AllImages/202407100950.jpg
Deleted: ../Dataset/AllImages/202407300920.jpg
Deleted: ../Dataset/AllImages/202407301300.jpg
Deleted: ../Dataset/AllImages/202408160450.jpg
Deleted: ../Dataset/AllImages/202408160540.jpg
Deleted: ../Dataset/AllImages/202408290620.jpg
Deleted: ../Dataset/AllImages/202409051240.jpg
Deleted: ../Dataset/AllImages/202409071500.jpg
Deleted: ../Dataset/AllImages/202409081650.jpg
Deleted: ../Dataset/AllImages/202409101040.jpg
Deleted: ../Dataset/AllImages/202410031800.jpg
Deleted: ../Dataset/AllImages/202410041250.jpg
Deleted: ../Dataset/AllImages/202410110620.jpg

Deleted 19 corrupted images.
